# Verification register

One row per activity links the current requirement sheet to calculation code or a reviewed evidence record. **373 requirements** are preserved, including open TBD/TBR values. Calculation availability and requirement closure are separate measures.

`SCREEN_PASS` means an explicit numerical limit was met under the stated model assumptions. `CALCULATED` has no approved acceptance rule. `MISSING_INPUTS` and `EVIDENCE_REQUIRED` remain open. A requirement closes only when every activity has reviewed passing evidence and its open parameters have been resolved.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from hyperion.workspace import current_flight
from hyperion.verification import ROOT, activities, input_contracts

get_ipython().run_line_magic('matplotlib', 'inline')
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.2})
pd.set_option('display.max_colwidth', 100)

In [2]:
from hyperion.verification import evaluate, requirement_status, export
context = current_flight()
results, details = evaluate(context=context)
requirements = requirement_status(results)
display(pd.crosstab(results.domain, results.status))
display(requirements.status.value_counts().rename_axis('Requirement status').to_frame('Count'))

status,CALCULATED,EVIDENCE_REQUIRED,MISSING_INPUTS,SCREEN_FAIL,SCREEN_PASS
domain,,,,,
1 LV Functional,2,22,26,2,0
2 LV Performance,0,35,19,1,4
3 Avionics,0,57,24,0,0
3 Manufacturing,0,15,0,0,0
3 Operations,0,134,51,0,0
3 Propulsion,4,81,21,2,2
3 Structural,0,81,76,0,0
Tier 0,1,6,10,1,0


,Count
Requirement status,
OPEN,367
FAILED_OR_INVALID,6


## Failed screens and missing inputs

Select a requirement to see its exact activity and source reference. `data/activities.csv` is the full index. There is no automatic acceptance of an activity just because its code exists or its source sheet lists an artifact.

In [3]:
display(results[results.status.isin(['SCREEN_FAIL','INVALID_INPUTS','INVALID_EVIDENCE'])][
    ['activity_id','activity','status','detail']])
REQUIREMENT = 'PERF.2.13'
display(results[results.requirement_id.eq(REQUIREMENT)][
    ['activity_id','activity','kernel','status','detail','source']])

,activity_id,activity,status,detail
0,MIS.0.01.A01,Integrated trajectory simulation from launch to TMI.,SCREEN_FAIL,600s coast and extra1.5s-equivalent restart debit are screening assumptions. Shared 6DOF prograd...
22,SYS.1.02.A01,"Integrated ascent, coast and finite-burn TMI simulation.",SCREEN_FAIL,600s coast and extra1.5s-equivalent restart debit are screening assumptions. Shared 6DOF prograd...
37,SYS.1.09.A02,Integrated finite-burn trajectory simulation.,SCREEN_FAIL,600s coast and extra1.5s-equivalent restart debit are screening assumptions. Shared 6DOF prograd...
85,PERF.2.13.A02,Control and aerodynamic moment balance.,SCREEN_FAIL,Requirement numeric screen only; model-validation and open source conflicts remain.
291,PROP.3.SYS.04.A02,First-stage engine thrust summation.,SCREEN_FAIL,3 source Raptor3 engines at2255kN each; source sea-level basis remains to be substantiated.
390,PROP.3.TANK.02.A01,Ullage and loaded-liquid volume calculations.,SCREEN_FAIL,"Current Mlox-Hlox Vehicle Sizing G4:H5,F19:G20,B31:C32; Vehicle Mass E29/E44 totals plus fairing..."


,activity_id,activity,kernel,status,detail,source
84,PERF.2.13.A01,Actuator limit and mass-property inspection.,NaN,EVIDENCE_REQUIRED,"Import the activity-specific CAD/FEA, specialist-model, inspection, demonstration or test record...",2 LV Performance!H14
85,PERF.2.13.A02,Control and aerodynamic moment balance.,flight_summary,SCREEN_FAIL,Requirement numeric screen only; model-validation and open source conflicts remain.,2 LV Performance!H14


## Add evidence or change a study

1. Find the stable activity ID and calculation in the register.
2. Add its required SI inputs and provenance to `config/verification_inputs.json`. Add explicit acceptance rules only where a sourced limit exists.
3. Run the relevant notebook or `python -m hyperion.verify`.
4. For tests, inspections or specialist models, supply the reviewed file and record described in `evidence/README.md`.

After later OneDrive changes, confirm sync and run `scripts/import_requirements.py` against the new workbook. Review configuration conflicts before regenerating evidence. The importer never edits the workbook.

In [4]:
output = export(results, details, ROOT/'results/current')
print(f'Wrote {len(results)} activity results and {len(requirements)} requirement statuses to results/current.')

Wrote 677 activity results and 373 requirement statuses to results/current.
